# 08 · Comparar el número de resultados k con nomic-embed-text en Qdrant

Medimos cuántos resultados (k = 1, 2, 4 y 8) conviene recuperar. Se fijan el modelo (`nomic-embed-text`, Ollama), la base (Qdrant) y la colección `pydocs_nomic_est1500_m200_sinruta` (troceo por estructura 1500/200, sin ruta de títulos), la decisión vigente de `docs/FICHA_05_CHUNK_SIZE.md`. No se indexa nada: el notebook solo consulta. Se evalúa con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md` y las evidencias de `docs/golden_set_evidencias.json`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`), Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`) y la colección creada con `notebooks/07_comparar_troceo_estructura.ipynb`.

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

`KS` es el único sitio donde se declaran los valores de k. El nombre de la colección sale de `MODELO`, `MAXIMO`, `MINIMO` y `CON_RUTA`, con el formato de `nombre_coleccion` del notebook 07.

In [ ]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from qdrant_client import QdrantClient
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, fuentes_ausentes, leer_evidencias, leer_golden_set,
    leer_k_minimo, preparar_query, rango_primer_acierto, rango_primer_acierto_evidencia,
    trocear_por_estructura,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "numero_k.csv"
NOTEBOOK_INDEXADO = "notebooks/07_comparar_troceo_estructura.ipynb"
MODELO = "nomic"
MAXIMO = 1500
MINIMO = 200
CON_RUTA = False
KS = (1, 2, 4, 8)


def nombre_coleccion(modelo: str, maximo: int, minimo: int, con_ruta: bool) -> str:
    sufijo = "ruta" if con_ruta else "sinruta"
    return f"pydocs_{modelo}_est{maximo}_m{minimo}_{sufijo}"


COLECCION = nombre_coleccion(MODELO, MAXIMO, MINIMO, CON_RUTA)
print("Colección:", COLECCION)

## 3. Conexión con Qdrant y Ollama

Se comprueba antes de calcular nada, para fallar pronto si falta un servicio. Si Qdrant u Ollama no responden, la celda para con un mensaje que dice qué hacer.

In [ ]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."
    ) from error

modelo_ollama = MODELOS[MODELO]["nombre"]
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."
    ) from error

print("Qdrant:", [col.name for col in qdrant.get_collections().collections])
print("Ollama:", modelo_ollama, "disponible")

## 4. Comprobación de la colección

El notebook no crea, borra ni modifica colecciones. Para antes de evaluar si la colección no existe o si su número de puntos no coincide con el de chunks del troceo por estructura.

In [ ]:
if not qdrant.collection_exists(COLECCION):
    raise RuntimeError(f"La colección {COLECCION} no existe. Créala con `{NOTEBOOK_INDEXADO}`.")

puntos = qdrant.count(COLECCION, exact=True).count
n_chunks = len(trocear_por_estructura(CARPETA_CORPUS, MAXIMO, MINIMO))
if puntos != n_chunks:
    raise RuntimeError(
        f"{COLECCION} tiene {puntos} puntos y el troceo {MAXIMO}/{MINIMO} da {n_chunks} chunks. "
        f"No la toco: recréala con `{NOTEBOOK_INDEXADO}`."
    )
print(f"{COLECCION}: {puntos} puntos, igual que los chunks del troceo")

## 5. Preguntas, evidencias y «k mín.» del golden set

Se leen de `docs/GOLDEN_SET_PYTHON_DOCS.md` y `docs/golden_set_evidencias.json`; nada está copiado en el código. El «k mín.» se lee del título de cada categoría y se conserva como texto. Se avisa, con el ID y la ruta, de las fuentes que no existen en el corpus.

In [ ]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

evidencias = leer_evidencias(RUTA_EVIDENCIAS)
k_minimo = leer_k_minimo(RUTA_GOLDEN)
print("k mín. declarado:", k_minimo)

## 6. Evaluación por k

Para cada k y cada pregunta con fuentes (A-D) se hace una búsqueda propia con `limit=k` (no se recorta la de k=8, porque HNSW es aproximado). Se guardan el rango por fuente, el rango por evidencia, los caracteres de contexto (suma de la longitud del `texto` de los chunks devueltos) y la latencia (embedding de la query más búsqueda).

In [ ]:
def embeber(texto: str) -> list[float]:
    return ollama.embed(model=modelo_ollama, input=[texto]).embeddings[0]


preguntas_abcd = [p for p in preguntas if p["fuentes"]]
resultados = []
for k in KS:
    for p in tqdm(preguntas_abcd, desc=f"k={k}"):
        inicio = time.perf_counter()
        vector = embeber(preparar_query(MODELO, p["query"]))
        res = qdrant.query_points(COLECCION, query=vector, limit=k, with_payload=True).points
        latencia_ms = (time.perf_counter() - inicio) * 1000
        recuperados = [{"fuente": r.payload["fuente"], "texto": r.payload["texto"]} for r in res]
        resultados.append({
            "k": k, "id": p["id"], "categoria": p["categoria"],
            "rango": rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"]),
            "rango_evidencia": rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]]),
            "contexto": sum(len(r["texto"]) for r in recuperados),
            "latencia_ms": latencia_ms,
        })

df = pd.DataFrame(resultados)
print(len(df), "evaluaciones")

## 7. Métricas por k (preguntas A-D)

hit@1, hit@k y MRR@k con los dos criterios: por fuente y por evidencia (más estricto, prefijo `ev_`). Las claves `hit@{k}` de `calcular_metricas` se renombran a `hit@k` para que todas las filas tengan las mismas columnas.

In [ ]:
def metricas(grupo: pd.DataFrame, k: int) -> dict:
    """Métricas por fuente y por evidencia (prefijo `ev_`), con nombres de columna iguales para todo k."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        m = calcular_metricas(rangos, k)
        filas.update({prefijo + "hit@1": m["hit@1"], prefijo + "hit@k": m[f"hit@{k}"], prefijo + "MRR": m["mrr"]})
    return filas


por_k = pd.DataFrame({k: metricas(df[df["k"] == k], k) for k in KS}).T
por_k

## 8. Desglose por categoría con el «k mín.»

hit@k y ev_hit@k por k y categoría, junto al «k mín.» que declara el golden set. Las categorías sin «k mín.» aparecen como «sin declarar».

In [ ]:
filas_cat = []
for k in KS:
    for categoria, grupo in df[df["k"] == k].groupby("categoria"):
        m = metricas(grupo, k)
        filas_cat.append({
            "k": k, "categoria": categoria, "k mín.": k_minimo.get(categoria, "sin declarar"),
            "preguntas": len(grupo), "hit@k": m["hit@k"], "ev_hit@k": m["ev_hit@k"],
        })
por_categoria = pd.DataFrame(filas_cat)
por_categoria.round(3)

## 9. Tabla resumen y CSV

Una fila por k, con las mismas columnas para todos. Se guarda en `data/resultados/numero_k.csv` (la carpeta se crea si no existe). Si hubiera una celda vacía, la celda falla.

In [ ]:
filas = []
for k in KS:
    sub = df[df["k"] == k]
    fila = {"k": k, **{n: v for n, v in por_k.loc[k].items()},
            "contexto medio (caracteres)": sub["contexto"].mean(),
            "latencia media (ms)": sub["latencia_ms"].mean()}
    for categoria in sorted(df["categoria"].unique()):
        m = metricas(sub[sub["categoria"] == categoria], k)
        fila[f"hit@k {categoria}"] = m["hit@k"]
        fila[f"ev_hit@k {categoria}"] = m["ev_hit@k"]
    filas.append(fila)
resumen = pd.DataFrame(filas)
assert not resumen.isna().any().any(), "El CSV tendría celdas vacías"
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")
resumen.round(3)